# 20a. 최종 모델 **학습** (M_maha_free)

**라벨을 쓰지 않음** — unlabeled 정상 생산 샷으로 '정상 공정 패턴'만 학습

1. reference 생성: unlabeled(2020-09-01~11-30) → labeled 시간대·이상 기록·짧은 Run·0 모드 제거
2. 파생변수 17개 계산 (같은 Run 과거 샷만 사용)
3. 학습: 변수별 중앙값·IQR (스케일) + Ledoit-Wolf 공분산 (변수들이 정상일 때 같이 움직이는 방식)
4. 저장: `outputs/final_model/{cn7,rg3}_final_model.pkl`

다음 단계 → `20b_final_test.ipynb`

In [1]:
import pandas as pd
from final_model import train, FEATURES
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

T = train()

[cn7] 학습 완료: reference 5,179샷, 변수 17개 → ./outputs/final_model/cn7_final_model.pkl, cn7_train_features.csv
[rg3] 학습 완료: reference 4,359샷, 변수 17개 → ./outputs/final_model/rg3_final_model.pkl, rg3_train_features.csv


## 1. 학습 데이터 처리 기록 (단계별 샷 수)

In [2]:
display(T['data_log'].set_index('제품').T)

제품,cn7,rg3
unlabeled 샷,28744,29633
labeled 시간대 제거·기간 적용 후,18068,4441
이상 기록 제거,1634,0
0 모드 제거,11174,47
reference 샷,5179,4359
reference Run,4,6
기준값 출처,reference_scales.json,reference_scales.json
학습 샷(최종),5179,4359
Ledoit-Wolf 축소계수,0.028782,0.017967


## 2. 학습된 파라미터
- **중앙값·IQR**: 정상 샷 기준 스케일 — 새 샷은 (값 − 중앙값) / IQR 로 변환
- **Ledoit-Wolf 축소계수**: 공분산을 얼마나 단순화(대각 방향으로 축소)했는지 (0 = 표본 공분산 그대로, 1 = 완전 축소)

In [3]:
display(T['params'].round(4))

,제품,변수,중앙값,IQR(스케일)
0,cn7,dev_Max_Injection_Speed,0.0000,1.3490
1,cn7,dev_Filling_Time,0.0000,1.3490
2,cn7,dev_Max_Injection_Pressure,0.0000,0.8993
3,cn7,dev_Max_Switch_Over_Pressure,0.0000,1.3490
4,cn7,dev_Average_Back_Pressure,0.0000,1.3490
5,cn7,dev_Cushion_Position,0.0000,0.4692
6,cn7,dev_Plasticizing_Time,0.0000,1.3490
7,cn7,dev_Cycle_Time,0.0000,4.0387
8,cn7,fill_anomaly,0.0000,2.6980
9,cn7,ev_max_abs_dev,1.6862,1.1510


## 3. 정상 샷끼리의 상관 (학습된 공분산 → 상관행렬)

In [4]:
import numpy as np
for p, m in T['models'].items():
    C = m.lw.covariance_
    d = np.sqrt(np.diag(C))
    corr = pd.DataFrame(C / np.outer(d, d), index=m.cols, columns=m.cols)
    print(p.upper(), '— 상관이 큰 변수 쌍 (|r| ≥ 0.3)')
    pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
    display(pairs[pairs.abs() >= 0.3].sort_values(key=abs, ascending=False).round(3).to_frame('r'))

CN7 — 상관이 큰 변수 쌍 (|r| ≥ 0.3)


r
dev_Max_Injection_Speed      dev_flow_res                 -0.929
                             fill_anomaly                 -0.919
fill_anomaly                 dev_flow_res                  0.910
dev_Filling_Time             fill_anomaly                  0.900
                             dev_flow_res                  0.814
dev_Max_Injection_Speed      dev_Filling_Time             -0.802
dev_Max_Switch_Over_Pressure fill_anomaly                  0.728
dev_Filling_Time             dev_Max_Switch_Over_Pressure  0.702
dev_Max_Switch_Over_Pressure dev_flow_res                  0.702
dev_Max_Injection_Speed      dev_Max_Switch_Over_Pressure -0.699
dev_Filling_Time             dev_Max_Injection_Pressure    0.686
dev_Max_Injection_Pressure   fill_anomaly                  0.670
                             dev_flow_res                  0.667
dev_Cycle_Time               ev_max_abs_dev                0.650
dev_Max_Injection_Speed      dev_Max_Injection_Pressure   -0.609
dev_Mold_Temperature_3       mold_slope20                  0.576
dev_Average_Back_Pressure    fill_anomaly                  0.571
dev_Max_Injection_Speed      dev_Average_Back_Pressure    -0.562
dev_Average_Back_Pressure    dev_flow_res                  0.553
dev_Max_Injection_Pressure   dev_Max_Switch_Over_Pressure  0.546
dev_Filling_Time             dev_Average_Back_Pressure     0.534
dev_Max_Switch_Over_Pressure dev_Average_Back_Pressure     0.520
mold_rise                    hopper_rise                   0.443
dev_Max_Injection_Pressure   dev_Average_Back_Pressure     0.440
dev_Max_Switch_Over_Pressure dev_Cushion_Position         -0.317
dev_Hopper_Temperature       hopper_slope20                0.301

RG3 — 상관이 큰 변수 쌍 (|r| ≥ 0.3)


r
dev_Max_Injection_Speed      fill_anomaly               -0.889
                             dev_flow_res               -0.703
fill_anomaly                 dev_flow_res                0.665
dev_Filling_Time             fill_anomaly                0.646
dev_Max_Injection_Pressure   dev_flow_res                0.594
mold_rise                    hopper_rise                 0.592
dev_Mold_Temperature_3       mold_slope20                0.587
dev_Max_Injection_Pressure   fill_anomaly                0.552
dev_Max_Injection_Speed      dev_Max_Injection_Pressure -0.494
dev_Max_Switch_Over_Pressure ev_max_abs_dev             -0.448
dev_Filling_Time             dev_Max_Injection_Pressure  0.414
mold_rise                    mold_slope20                0.361
dev_Cycle_Time               ev_max_abs_dev              0.347
dev_Cushion_Position         fill_anomaly               -0.332
dev_Max_Injection_Speed      dev_Filling_Time           -0.321
dev_Filling_Time             dev_Cushion_Position       -0.315

## 4. 정상 샷 점수 분포 → 임계값
q = 0.999 → 정상 샷의 0.1%만 넘는 점수 (이 값을 넘으면 검사 대상)

In [5]:
rows = []
for p, m in T['models'].items():
    for q in (0.95, 0.99, 0.995, 0.999):
        rows.append({'제품': p, 'q': q, '정상 오경보율': f'{100 * (1 - q):g}%', '점수 임계값': m.threshold(q)})
display(pd.DataFrame(rows).pivot(index='q', columns='제품', values='점수 임계값').round(3))

제품,cn7,rg3
q,,
0.950,4.994,5.574
0.990,10.736,9.835
0.995,12.296,12.437
0.999,20.841,15.575
